# scTASL tutorial: cell-type-specific GRN inference

This notebook continues from `1_integration_fine-tuning.ipynb`. It loads the
cell-type-specific fine-tuned RNA/ATAC embeddings from
`results/<DATASET>/GRN/<cell_type>/adata/` and infers a gene regulatory network
(GRN) for the selected cell type.

The workflow consists of the following steps:

1. Load fine-tuned RNA/ATAC embeddings and the prior gene-peak graph.
2. Score candidate gene-peak links by feature-embedding similarity and FDR.
3. Retain significant gene-peak links as cis-regulatory evidence.
4. Identify candidate transcription factors (TFs) using motif annotations.
5. Build a peak-to-TF motif-overlap graph.
6. Rank gene-TF cis-regulatory associations.
7. Infer a draft TF-to-target network with GRNBoost2.
8. Prune and score the final GRN using co-expression, embedding, and cis evidence.
9. Optionally plot selected TF-target subnetworks from the pruned GRN.

> Run `1_integration_fine-tuning.ipynb` first for the same `TARGET_CELL_TYPE`.
> Only load pickle files generated by this project or another trusted source.


## 0. Setup

### 0.1 Import libraries

In [ ]:
import os
import pickle

import anndata
import numpy as np
import pandas as pd
import matplotlib as mpl

from utils.grn.functions import (
    Bed,
    regulatory_inference,
    edge_subgraph,
    cis_regulatory_ranking,
)
from utils.grn.inference_utils import (
    apply_importance_filter,
    filter_peak2tf_to_candidate_tfs,
    load_or_compute_peak2tf,
    plot_selected_tf_network,
    print_pruning_config,
    save_and_summarize_pruned_grn,
    score_and_prune_grn,
)

# Keep text editable when PDF figures are opened in vector-graphics software.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42


### 0.2 Configure dataset and paths

Set `TARGET_CELL_TYPE` to the same value used in
`1_integration_fine-tuning.ipynb`. This notebook reads the fine-tuned embeddings
from `results/<DATASET>/GRN/<SAFE_CELL_TYPE>/adata/` and writes GRN inference
outputs to `results/<DATASET>/GRN/<SAFE_CELL_TYPE>/inference/`.


In [ ]:
DATASET = "PBMC-10k"
TARGET_CELL_TYPE = "CD14_Mono"
SAFE_CELL_TYPE = TARGET_CELL_TYPE.replace(" ", "_").replace("/", "_")

FINE_TUNE_DIR = f"./results/{DATASET}/GRN/{SAFE_CELL_TYPE}"
DATA_DIR = os.path.join(FINE_TUNE_DIR, "adata")
RESULT_DIR = os.path.join(FINE_TUNE_DIR, "inference")
PRIOR_DIR = f"./dataset/{DATASET}"
ANNOT_DIR = "./dataset/annotation_files"

os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Dataset: {DATASET}")
print(f"Target cell type: {TARGET_CELL_TYPE}")
print(f"Fine-tuning directory: {FINE_TUNE_DIR}")
print(f"Embedding input directory: {DATA_DIR}")
print(f"GRN output directory: {RESULT_DIR}")
print(f"Prior graph directory: {PRIOR_DIR}")
print(f"Annotation directory: {ANNOT_DIR}")


## 1. Load fine-tuned embeddings and the prior graph

This step loads the AnnData objects generated by
`1_integration_fine-tuning.ipynb`:

- `rna_finetuned_<SAFE_CELL_TYPE>.h5ad`
- `atac_finetuned_<SAFE_CELL_TYPE>.h5ad`

The prior graph `graph_data.pkl` is the same gene-peak graph used by the full
integration run that initialized fine-tuning.


In [ ]:
rna_path = os.path.join(DATA_DIR, f"rna_finetuned_{SAFE_CELL_TYPE}.h5ad")
atac_path = os.path.join(DATA_DIR, f"atac_finetuned_{SAFE_CELL_TYPE}.h5ad")
graph_path = os.path.join(PRIOR_DIR, "graph_data_grn.pkl")

missing_files = [
    path for path in (rna_path, atac_path, graph_path) if not os.path.isfile(path)
]
if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s):\n- " + "\n- ".join(missing_files)
    )

rna = anndata.read_h5ad(rna_path)
atac = anndata.read_h5ad(atac_path)
with open(graph_path, "rb") as handle:
    graph_data = pickle.load(handle)

print(f"RNA: {rna.n_obs} cells x {rna.n_vars} genes")
print(f"ATAC: {atac.n_obs} cells x {atac.n_vars} peaks")
print(
    f"Prior graph: {graph_data.edge_index.shape[1]} edges, "
    f"{graph_data.num_nodes} nodes"
)


### 1.1 Combine feature embeddings

Genes and peaks are represented in one shared feature-embedding space. The
prior graph uses this unified feature index.


In [ ]:
features = pd.Index(np.concatenate([rna.var_names, atac.var_names]))
feature_embeddings = np.concatenate(
    [rna.varm["feature_embedding"], atac.varm["feature_embedding"]], axis=0
)

print(f"Unified feature space: {len(features)} features, "
      f"embedding dim = {feature_embeddings.shape[1]}")

## 2. Cis-Regulatory Inference

Score every candidate gene–peak edge in the prior graph using
**embedding cosine similarity** between the learned feature embeddings,
combined with a permutation-based FDR estimate.
Significant edges (q < 0.05) are retained as the structural evidence
for downstream GRN construction.

In [ ]:
# Step 2a: Embedding-based regulatory inference
# Scores every candidate edge with cosine similarity + permutation p-value
reginf = regulatory_inference(
    features,
    feature_embeddings,
    prior_graph=graph_data,
    random_state=1607,
)

print(f"Edges scored: {reginf.edge_index.shape[1]}")
print(f"Edge attributes: {list(reginf.edge_attr_dict.keys())}")

## 3. Significance filtering

Retain gene-peak edges with embedding FDR q-value below `QVAL_THRESHOLD`, then
remove isolated nodes. These significant gene-peak links are used as structural
cis-regulatory evidence in the later TF-target pruning step.


In [ ]:
# Retain edges where embedding FDR q-value < 0.05
QVAL_THRESHOLD = 0.05

qval      = np.asarray(reginf.edge_attr_dict["qval"])
mask      = qval < QVAL_THRESHOLD

g_sig = edge_subgraph(reginf, mask, drop_isolated=True, reindex_nodes=True)

print(f"Embedding q<{QVAL_THRESHOLD} : {mask.sum():>6d} edges retained")
print(f"Remaining nodes : {g_sig.num_nodes}  "
      f"({len(g_sig.gene_names)} genes, {len(g_sig.peak_names)} peaks)")


## 4. Identify transcription factors (TFs)

We use a JASPAR2022 motif-scan BED file to define candidate transcription
factors. Choose the motif annotation that matches the genome build of your
dataset: use the hg38 file for human data and the mm10 file for mouse data.

The TF names from the selected motif file are intersected with genes present
in the RNA AnnData object, so downstream GRN inference only uses TFs available
in the current dataset.

In [ ]:
motif_bed = Bed.read_bed(os.path.join(ANNOT_DIR, "JASPAR2022-hg38.bed.gz"))

tfs = pd.Index(motif_bed["name"]).intersection(rna.var_names)

print(f"JASPAR motif entries : {len(motif_bed):,}")
print(f"Candidate TFs (in RNA data) : {tfs.size}")
print(f"Examples: {', '.join(tfs[:10])}")

## 5. Build peak-to-TF mapping

Overlap ATAC peaks with JASPAR motif positions to create a peak-to-TF graph.
This step can be expensive, so the result is cached as `peak2tf.pkl`.

Set `RECOMPUTE_PEAK2TF = True` to rebuild the cache from scratch.


In [ ]:
genes = rna.var_names
peaks = atac.var_names

# Ensure ATAC var has a 'name' column for Bed constructor
atac.var["name"] = atac.var_names
peak_bed = Bed(atac.var.loc[peaks])

In [ ]:
PEAK2TF_CACHE = os.path.join(RESULT_DIR, "peak2tf.pkl")
RECOMPUTE_PEAK2TF = False

# Load peak2tf.pkl if it already exists. If it is missing, or if an old cache
# lacks motif_score, compute the peak-to-TF graph and save it to this path.
peak2tf, has_motif_score = load_or_compute_peak2tf(
    peak_bed=peak_bed,
    motif_bed=motif_bed,
    cache_path=PEAK2TF_CACHE,
    recompute=RECOMPUTE_PEAK2TF,
    require_motif_score=True,
)

print(
    f"peak2tf: {peak2tf.edge_index.shape[1]} edges, "
    f"{len(peak2tf.node_names)} nodes | "
    f"motif_score available: {has_motif_score}"
)


Filter the peak-to-TF graph to retain candidate TFs present in the RNA data.


In [ ]:
peak2tf_filtered = filter_peak2tf_to_candidate_tfs(peak2tf, tfs)

print(
    f"After TF filtering: {peak2tf_filtered.edge_index.shape[1]} edges "
    f"(from {peak2tf.edge_index.shape[1]})"
)


## 6. Cis-regulatory ranking

For each (gene, TF) pair, compute how many shared ATAC peaks link them
via the gene → peak → TF path, compared to a length-matched random
background. The result is a rank matrix: lower rank = stronger cis-regulatory
association.

In [ ]:
gene2tf_rank = cis_regulatory_ranking(
    g_sig,
    peak2tf_filtered,
    genes,
    peaks,
    tfs,
    region_lens=(atac.var.loc[peaks, "chromEnd"]
                 - atac.var.loc[peaks, "chromStart"]).tolist(),
    weight_attr="motif_score" if has_motif_score else None,
    random_state=0,
)

print(f"Gene x TF rank matrix: {gene2tf_rank.shape}")
print(f"motif_score weighting: {'enabled' if has_motif_score else 'disabled (binary fallback)'}")
gene2tf_rank.iloc[:5, :5]


## 7. Co-expression network (GRNBoost2)

Use [GRNBoost2](https://arboreto.readthedocs.io/) (stochastic gradient boosting)
to infer TF → target importance scores from the RNA expression matrix.

> **Note:** This step is computationally expensive. Adjust `N_WORKERS` for your machine.

In [ ]:
DRAFT_GRN_FILE = os.path.join(RESULT_DIR, "draft_grn_grnboost2.csv")
N_WORKERS = 20
RECOMPUTE_GRNBOOST = False  # Set True to re-run GRNBoost2.

if RECOMPUTE_GRNBOOST or not os.path.exists(DRAFT_GRN_FILE):
    from distributed import Client, LocalCluster
    from arboreto.core import create_graph
    from arboreto.algo import SGBM_KWARGS

    # Prepare expression matrix
    X = rna.X.toarray() if hasattr(rna.X, "toarray") else np.asarray(rna.X)
    exp = pd.DataFrame(X, index=rna.obs_names, columns=rna.var_names)

    # Launch Dask cluster
    cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=1, processes=True)
    client  = Client(cluster)
    print(f"Dask dashboard: {client.dashboard_link}")

    try:
        graph = create_graph(
            expression_matrix=exp.values.astype(np.float32, copy=False),
            gene_names=exp.columns.tolist(),
            tf_names=tfs.astype(str).tolist(),
            client=client,
            regressor_type="GBM",
            regressor_kwargs=SGBM_KWARGS,
            seed=0,
        )
        network = graph.compute()
        network.to_csv(DRAFT_GRN_FILE, index=False)
        print(f"Saved draft GRN: {len(network)} edges → {DRAFT_GRN_FILE}")
    finally:
        client.close()
        cluster.close()
else:
    network = pd.read_csv(DRAFT_GRN_FILE)
    print(f"Loaded cached draft GRN: {len(network)} edges from {DRAFT_GRN_FILE}")

network.head()

## 8. Prune and finalize the GRN

The draft GRNBoost2 network is pruned and scored in three steps:

1. Importance filtering keeps high-importance GRNBoost2 edges.
2. Cis-regulatory filtering keeps TF-target pairs supported by `gene2tf_rank`.
3. Peak support and embedding FDR evidence are aggregated into a final
   rank-based `final_reg_score`.


In [ ]:
PRUNE_MODE = "strict"

network_filtered, importance_threshold, prune_configs, prune_config = apply_importance_filter(
    network,
    PRUNE_MODE,
)
IMPORTANCE_QUANTILE = prune_config["importance_q"]
CIS_TOP_THRESHOLD = prune_config["cis_top"]
REQUIRE_SUPPORT_PEAK = prune_config["require_peak"]

print_pruning_config(
    mode=PRUNE_MODE,
    config=prune_config,
    prune_configs=prune_configs,
    importance_threshold=importance_threshold,
    n_before=len(network),
    n_after=len(network_filtered),
)


In [ ]:
FULL_GRN_FILE = os.path.join(RESULT_DIR, "full_grn.csv")

network_scored, network_pruned, n_sig_pairs = score_and_prune_grn(
    network_filtered=network_filtered,
    gene2tf_rank=gene2tf_rank,
    g_sig=g_sig,
    peak2tf_filtered=peak2tf_filtered,
    genes=rna.var_names,
    peaks=atac.var_names,
    tfs=tfs,
    cis_top_threshold=CIS_TOP_THRESHOLD,
    require_support_peak=REQUIRE_SUPPORT_PEAK,
    full_grn_file=FULL_GRN_FILE,
)

print(f"Significant cis-regulatory TF-target pairs: {n_sig_pairs:,}")
print(f"Full scored GRN before pruning: {len(network_scored):,} edges -> {FULL_GRN_FILE}")
print(
    f"After cis + support pruning: {len(network_filtered):,} -> "
    f"{len(network_pruned):,} edges"
)
print(
    f"Final score range: {network_pruned['final_reg_score'].min():.4f} ~ "
    f"{network_pruned['final_reg_score'].max():.4f}"
)


In [ ]:
OUTPUT_FILE = os.path.join(RESULT_DIR, "pruned_grn.csv")
save_and_summarize_pruned_grn(network, network_pruned, OUTPUT_FILE)
network_pruned.head(30)


## 9. Plot selected TF-target subnetworks (optional)

This visualization reads `pruned_grn.csv` and does not change the inferred GRN.


In [ ]:
PRUNED_GRN_FILE = os.path.join(RESULT_DIR, "pruned_grn.csv")
PLOT_FILE = os.path.join(RESULT_DIR, "tf_gene_selected_network.png")
PLOT_PDF_FILE = os.path.join(RESULT_DIR, "tf_gene_selected_network.pdf")

TF_MAP = {
    "CD14_Mono": ["SPI1", "STAT1", "IRF1"],
    "CD4_Naive": ["ETS1", "ERG"],
    "Naive_B": ["BATF", "PAX5"],
}
SELECTED_TFS = TF_MAP.get(SAFE_CELL_TYPE, ["SPI1", "IRF1", "STAT1"])

TOP_GENES_PER_TF = 20
MIN_FINAL_SCORE = None  # Example: 0.8
EDGE_WIDTH = 1.8
LABEL_FONTSIZE = 12
EXPORT_DPI = 1200

grn = pd.read_csv(PRUNED_GRN_FILE)
plot_selected_tf_network(
    grn=grn,
    rna=rna,
    selected_tfs=SELECTED_TFS,
    plot_file=PLOT_FILE,
    plot_pdf_file=PLOT_PDF_FILE,
    top_genes_per_tf=TOP_GENES_PER_TF,
    min_final_score=MIN_FINAL_SCORE,
    edge_width=EDGE_WIDTH,
    label_fontsize=LABEL_FONTSIZE,
    export_dpi=EXPORT_DPI,
)
